In [1]:
import numpy as np
from sklearn.linear_model import Ridge
from itertools import chain
import pandas as pd
from scipy.stats import norm
from collections import defaultdict
import random
import matplotlib.pyplot as plt
import time
import inventory_control as IC

In [ ]:
H = 10 # Horizon length
lambda_reg = 0.0001  # Regularization

coef_GT = [-2,-4,-2, 5, 3, -11*(H-1)/8 - 23/8]
value_GT= IC.value_return_estimation_optimal_policy_uniform_init(coef_GT, "quadratic", H)
#print(f"Groundtrue return v_pi: {value_GT}")

shape_param=1
rate_param=0.1
H_bar = shape_param/rate_param 
n_episodes=400

#type = "quadratic"#"cubic"#
#n_components = 6

type = "quadratic"#"cubic"
n_components = 6#10

# Parameters for confidence interval calculation
confidence_level = 0.9  # 90% confidence level
z_value = norm.ppf(1 - (1 - confidence_level) / 2)  # Z-value for two-tailed test ##????

subset_size = int(np.floor(n_episodes ** 1))
B = 100

# Number of repetitions for empirical coverage calculation
n_repetitions = 100#10#80#10#15
covered_count_plugin = 0
covered_count_BS = 0
valid_loop_count = 0

rank_relaxation = 0

# Lists to store values for each repetition
v_pi_fqe_list = []
plugin_sigma_squared_avg_list = []
plugin_interval_width_list = []
plugin_CI_list = []

BS_var_list = []
BS_interval_width_list = []
BS_CI_list = []

#print(f"d H^3: {n_components * (H**3)}")
#print(f"Plugin Approx subset length: {n_episodes * H_bar/2}")
#print(f"BS Approx subset length: {subset_size * H_bar}")
plugin_time_list = []
BS_time_list     = []

#rank_relaxation = 5
output_file = f"splitting_plugin_BS_CIlevel_{confidence_level:.2f}_K_{n_episodes}_Hbar_{H_bar}_H_{H}_B_{B}_subsetsize_{subset_size}_type_{type}_1.csv"
for _ in range(n_repetitions):
    #print(rep)
    # Recalculate trajectories, Q-table, and returns to simulate repeated experiments
    trajectories = IC.data_generation_inventory_control(n_episodes,shape_param,rate_param)
    flatten_full_traj = list(chain.from_iterable(trajectories))
    N = len(flatten_full_traj)
    
    feature_table_t = np.vstack([IC.feature_base_func(transition['state'], transition['action'], type) for transition in flatten_full_traj])
    Sigma_0_est_traj = IC.compute_sigma_0_est(feature_table_t, n_episodes, H_bar)#compute_sigma_0_est(flatten_full_traj, features_table_t, n_actions, n_episodes, H_bar)
    rank_full = np.linalg.matrix_rank(Sigma_0_est_traj)
    print(f"Rank of Sigma0_est from full trajectory: {rank_full}")
    if rank_full < n_components - rank_relaxation:
        print("Skipping iteration due to singularity")
        continue
    
    t0 = time.time()
    # Split trajectories into two batches A1 and A2
    split_index = len(trajectories) // 2
    batch_A1 = trajectories[:split_index]
    batch_A2 = trajectories[split_index:]

    # Increment valid loop count
    valid_loop_count += 1
    
    theta_est,_ = IC.FQE_inventory_control(H, flatten_full_traj,lambda_reg,type)
    theta_0 = theta_est[:,0]
    value_fqe = IC.value_return_estimation_optimal_policy_uniform_init(theta_0, type, H)
    
    #variance_BS, confidence_interval_BS, epsilon_list_BS = subsampled_bootstrapping_fqe(trajectories, value_fqe, Q_H, confidence_level, subset_size, B,initial_state_distribution, target_policy, target_epsilon)

    # Compute sigma^2 for the first configuration (using batch_A1 for function estimation and batch_A2 for evaluation)
    sigma_squared_A2,epsilon_2, Omega_est_list_2 = IC.compute_sigma_squared(batch_A1, batch_A2, H, H_bar, lambda_reg, type)

    # Compute sigma^2 for the second configuration (using batch_A2 for function estimation and batch_A1 for evaluation)
    sigma_squared_A1,epsilon_1, Omega_est_list_1 = IC.compute_sigma_squared(batch_A2, batch_A1, H, H_bar, lambda_reg, type)

    # Average the two sigma^2 estimates
    sigma_squared_avg = (sigma_squared_A2 + sigma_squared_A1) / 2

    # Standard deviation for the confidence interval
    #std_error = np.sqrt(sigma_squared_avg / n_episodes)
    std_error = np.sqrt(sigma_squared_avg / N)

    # Construct confidence interval
    lower_bound_plugin = value_fqe - z_value * std_error
    upper_bound_plugin = value_fqe + z_value * std_error

    # Calculate interval width
    interval_width_plugin = upper_bound_plugin - lower_bound_plugin

    # Store values for each repetition
    v_pi_fqe_list.append(value_fqe)
    plugin_sigma_squared_avg_list.append(sigma_squared_avg)
    plugin_interval_width_list.append(interval_width_plugin)
    plugin_CI_list.append([lower_bound_plugin,upper_bound_plugin])

    # Check if the ground truth value lies within the confidence interval
    if lower_bound_plugin <= value_GT <= upper_bound_plugin:
        covered_count_plugin += 1
        
    plugin_time = time.time() - t0
    plugin_time_list.append(plugin_time)
        
    #subsampled bootstrap
    t1 = time.time()
    variance_BS, confidence_interval_BS, epsilon_list_BS = IC.subsampled_bootstrapping_fqe(trajectories, value_fqe, confidence_level, subset_size, B,lambda_reg, H,type)
    
    BS_var_list.append(variance_BS)
    BS_interval_width_list.append(confidence_interval_BS[1] - confidence_interval_BS[0])
    BS_CI_list.append(confidence_interval_BS)

    # Check if the ground truth value lies within the confidence interval
    if confidence_interval_BS[0] <= value_GT <= confidence_interval_BS[1]:
        covered_count_BS += 1
        
    bs_time = time.time() - t1
    BS_time_list.append(bs_time)

    plugin_empirical_coverage = covered_count_plugin / valid_loop_count
    BS_empirical_coverage = covered_count_BS / valid_loop_count

    # Align lengths by padding with None
    max_length = max(len(v_pi_fqe_list), len(plugin_sigma_squared_avg_list), len(plugin_interval_width_list), len(BS_interval_width_list), len(BS_var_list), len(BS_time_list), len(plugin_time_list))
    aligned_data = {
        "value_GT": [value_GT] + [None] * (max_length - 1),  
        "v_pi_fqe": v_pi_fqe_list + [None] * (max_length - len(v_pi_fqe_list)),
        "plugin_sigma_squared_avg": plugin_sigma_squared_avg_list + [None] * (max_length - len(plugin_sigma_squared_avg_list)),
        "plugin_empirical_coverage": [plugin_empirical_coverage] + [None] * (max_length - 1),
        "plugin_interval_width": plugin_interval_width_list + [None] * (max_length - len(plugin_interval_width_list)),
        "plugin_confidence_interval": plugin_CI_list + [None] * (max_length - len(plugin_CI_list)),
        "plugin_covered_count": [covered_count_plugin] + [None] * (max_length - 1),
        "plugin_time_sec": plugin_time_list + [None]*(max_length-len(plugin_time_list)),
        "BS_variance": BS_var_list + [None] * (max_length - len(BS_var_list)),
        "BS_empirical_coverage": [BS_empirical_coverage] + [None] * (max_length - 1),
        "BS_interval_width": BS_interval_width_list + [None] * (max_length - len(BS_interval_width_list)),
        "BS_confidence_interval": BS_CI_list + [None] * (max_length - len(BS_CI_list)),
        "BS_covered_count": [covered_count_BS] + [None] * (max_length - 1),
        "BS_time_sec": BS_time_list + [None]*(max_length-len(BS_time_list)),
        "valid_loop_count": [valid_loop_count] + [None] * (max_length - 1)
    }


    df = pd.DataFrame(aligned_data)
    df.to_csv(output_file, index=False)

print(f"Results saved to {output_file}")
print(df)

Rank of Sigma0_est from full trajectory: 6
Rank of Sigma0_est from full trajectory: 6
